Broadcasting: The term broadcasting describes how Numpy treats arrays with different shapes during arithmetic operations. They are subjected to certain constraints, the smaller array is "broadcast" across the larger array so that they have compatible shapes.

Broadcasting - provides a means of vectorising array opeartions so that looping - occurs in C rather than in Python. Does without making a copy of data and leads to effiecient algorithm implementations.

There are however some cases where Broadcasting is a bad idea as it leads to ineffiecient use of memory that slows computation.
















In numpy we do operations on pair of arrays on an element-by-element basis.

simpleast case: array have same shap but the broadcasting rules are used when the arrays shape meets certain constraints.
Simplest broadcasting == when an array and a scaler values are combined in an operation. Here we can think it as:

Imagine that the scaler b is being stretched during an arithmetic operation into an array with the same shape as a. The new elements are simply copies of the scalar b.

The stretching is conceptual only! Numpy is so smart that it used the og scalar value without actually making copies so that: Broadcasting operation == More memory and computationally efficient as possible !

GENERAL BROADCASTING RULE:
When operating two arrays. Numpy compares their shape element wise. Starts with trailing rightmost dimension and works its way left.

Two D's are only compatible when:
1. they are equal in shape
2. one of them is 1.

And this is looks so easy but Numpy sometimes creates error while broadcasting.

If these conditions are not met a `ValueError: oparands could not be broadcast together` exception is thrown: indicates array shapes are incompatible.

Input arrays do not need to have the same number of dimensions.

The resulting array will have the same number of dimensions as the input array with the greatest number of dimensions, where the size of each dimension is the largest size of the corresponding dimension among the input array.

R_Array == same no. of dimensions as input array with greatest no. of dimension:
size fo each dimension == largest size of corresponding dimension among the input array.

When either of the dimensions compared is one, the other is used. In other words, dimensions with size 1 are stretched or “copied” to match the other.

In [2]:
import numpy as np

In [20]:
# Trap 1: (100,3) matrix. Subtract column means of shape (3,). Then try means.reshape(1,3) vs means.reshape(3,1). Which is correct? Verify by checking zero mean per column.

array = np.linspace(1, 100, 300).reshape((100,3))
column_mean = array.mean(axis=0,keepdims = True)
# a pro data scientist uses keepdims = True so that the array maintains itself.
# to enforce broadcasting i must reshape the array into a 2d one
# column_mean = column_mean.reshape((3,1))
print(column_mean)
print((array-column_mean).mean(axis=0))

[[50.16889632 50.5        50.83110368]]
[ 1.90425453e-14 -1.84741111e-15 -1.27897692e-14]


In [25]:
# Trap 2: Two arrays (5,) and (5,). np.dot gives scalar. Reshape to (5,1) and (1,5). What does @ give now? Predict first.

# Prediction: @ will give (5,1) @ (1,5 )==( 5,5)
# but i do think that in numpy np.dot and @ follows same matrix multiplication i can be wrong and the answer still might be a scalar but we have reshaped it meaning yahhh ! It will be 5,5
array1 = np.arange(5)
array2 = np.arange(5)

print(np.dot(array1, array2))

array1 = array1.reshape(5,1)
array2 = array2.reshape(1,5)

print(array1 @ array2, (array1@array2).shape)

# prediction was right

30
[[ 0  0  0  0  0]
 [ 0  1  2  3  4]
 [ 0  2  4  6  8]
 [ 0  3  6  9 12]
 [ 0  4  8 12 16]] (5, 5)


In [33]:
# Trap 3: Add bias (3,) to batch outputs (32,3). Try it. Now try shape (3,1) instead. What shape did you actually add?

array1 = np.linspace(1,10, 96).reshape((32,3))
array2 = np.arange(3)

# Prediction: If i add 3, to 32,3 i will still have 32,3
# and if i reshaped 3, to 3,1 and then add to 32,1 i will get 32,32

#Let's see if my predictions are actually correct.

sum1 = array1 + array2 # here (3,) is treated by numpy as (1,3) to match the 2 dimn of (32,3)
print(sum1[0:5,:])
# sum2 = array1 + array2.reshape((3,1))

# Shit i was wrong ! Lesson Learned: It leads to value error as (32,3) and (3,1)
# 3 and 1 are rightmst so 1 can stretch to make itself 3, ut when we come to 100 and 3 100 != 3 and none of them is one and here is where the broadcating failed

[[1.         2.09473684 3.18947368]
 [1.28421053 2.37894737 3.47368421]
 [1.56842105 2.66315789 3.75789474]
 [1.85263158 2.94736842 4.04210526]
 [2.13684211 3.23157895 4.32631579]]


In [ ]:
# 4. Pairwise squared distances between 100 points. Loop vs broadcasting with (X - X.reshape(100,1,2)). What shape?

First, let's create our 100 points, each with 2 coordinates. We'll use `np.random.rand` for demonstration purposes.

In [35]:
import numpy as np

# Create 100 points, each with 2 coordinates (e.g., X, Y)
X = np.random.rand(100, 2)

print("Shape of X:", X.shape)


Shape of X: (100, 2)


Now, let's use the broadcasting trick to get all pairwise differences. The hint `(X - X.reshape(100,1,2))` is crucial here. Let's break down what's happening:

*   `X` has shape `(100, 2)`.
*   `X.reshape(100, 1, 2)` changes `X` into an array of shape `(100, 1, 2)`. This means we have 100 'rows', each containing a single 'point' (with 2 coordinates).

When we subtract `X` (effectively broadcast to `(100, 100, 2)`) from `X.reshape(100, 1, 2)` (effectively broadcast to `(100, 100, 2)`), the result `diffs` will have a shape of `(100, 100, 2)`.

Each element `diffs[i, j, :]` will represent the vector difference `X[i, :] - X[j, :]`.


In [38]:
# Calculate pairwise differences
# X has shape (100, 2)
# X.reshape(100, 1, 2) has shape (100, 1, 2)
# When subtracted, X is broadcast from (100, 2) to (100, 100, 2)
# and X.reshape(100, 1, 2) is broadcast from (100, 1, 2) to (100, 100, 2)
diffs = X - X.reshape(100, 1, 2)
print("Shape of pairwise differences (diffs):", diffs.shape)
print("Example: Difference between point 0 and point 1 (first 5 values):\n", diffs[0, 1, :])
print("Expected: X[0] - X[1]:\n", X[0] - X[1])

Shape of pairwise differences (diffs): (100, 100, 2)
Example: Difference between point 0 and point 1 (first 5 values):
 [0.02887018 0.2053761 ]
Expected: X[0] - X[1]:
 [-0.02887018 -0.2053761 ]


Finally, to get the squared Euclidean distance, we square these differences and sum them along the last axis (the coordinate axis).

In [39]:
# Square the differences
squared_diffs = diffs**2

# Sum along the last axis (axis=2) to get the squared distance
pairwise_squared_distances = np.sum(squared_diffs, axis=2)

print("Shape of pairwise squared distances:", pairwise_squared_distances.shape)
print("First 5x5 block of pairwise squared distances:\n", pairwise_squared_distances[:5, :5])

# As a check, the diagonal elements should be 0 (distance of a point to itself)
print("Diagonal elements (should be close to 0):\n", np.diag(pairwise_squared_distances[:5, :5]))

Shape of pairwise squared distances: (100, 100)
First 5x5 block of pairwise squared distances:
 [[0.         0.04301283 0.56492029 0.0458574  0.1537378 ]
 [0.04301283 0.         0.84733325 0.02693468 0.3469337 ]
 [0.56492029 0.84733325 0.         0.63532992 0.44561377]
 [0.0458574  0.02693468 0.63532992 0.         0.35391787]
 [0.1537378  0.3469337  0.44561377 0.35391787 0.        ]]
Diagonal elements (should be close to 0):
 [0. 0. 0. 0. 0.]


Finding it hard to do intuitively understand pairwise distance

In [55]:
# Trap 5: Normalise matrix (100,50) row-wise. Norms have shape (100,). Does data / norms work? What reshape makes it work?

# for normalisation we first calculate the noorm of the array and then divide the whole array with the norm.

array = np.random.rand(100,50)
print(array.shape)

norm_array = np.linalg.norm(array,axis =1) # Row wise!
print(norm_array.shape)

# the reshape that makes data / norm work is reshape(100,1)
# as from rightmost: 1 will be strecthed to 50 and 100 is compatible to 100 only

# 1. I will try without reshaping
try:
  normalised_array = array / norm_array
except ValueError:
  print("Can't broadcast 100,5 wih 100, as for 100, numpy tries to make it a match with 100,5 but it can't that's why: reshape to a broadcastable shape")

# 2. let us reshape it to predicted 100,1 reshaped array so it will work then
try:
  norm_array = norm_array.reshape((100,1))
  normalised_array = array / norm_array
  print("Normalisation compatible with 100,50 and 100,1 array:", normalised_array[0:5,0:5])
except ValueError:
  print("Can't broadcast reshape to a broadcastable shape")

(100, 50)
(100,)
Can't broadcast 100,5 wih 100, as for 100, numpy tries to make it a match with 100,5 but it can't that's why: reshape to a broadcastable shape
Normalisation compatible with 100,50 and 100,1 array: [[0.00466544 0.02588401 0.00453746 0.09811501 0.18082165]
 [0.16387141 0.15695064 0.2421801  0.19301917 0.21024909]
 [0.08094851 0.14739354 0.12217666 0.22143982 0.06011645]
 [0.1486529  0.04332747 0.15954505 0.22089015 0.19440104]
 [0.12443211 0.14630094 0.09042088 0.04264689 0.01456688]]


Matirx vs Element wise Multiplication

Universal Functions Applications

predict Shapes and Types of the data with gemini made drills
figure out common traps in broadcasting